# Gemma 4 Particle Edu
### Free 3D Physics Simulation Platform powered by Gemma 4 + Ollama

**Tracks**: Future of Education (Impact) / Ollama Special Technology / Main

| | |
|---|---|
| Live Demo | [gemma4-particle-edu.vercel.app](https://gemma4-particle-edu.vercel.app) |
| Gemma 4 Demo | [`?model=gemma4`](https://gemma4-particle-edu.vercel.app/?model=gemma4) |
| Video | [YouTube (3 min)](https://youtu.be/3e-LZPHBA2M) |
| GitHub | [U2SY26/gemma4-particle-edu](https://github.com/U2SY26/gemma4-particle-edu) |
| Benchmark Data | [Kaggle Dataset](https://www.kaggle.com/datasets/syu21125/gemma4-particle-edu-benchmark-300) |
| Real-World Deployment | [Visual Science Lab — Google Play](https://play.google.com/store/apps/details?id=com.sciencelab.science_lab_flutter) (8,470 installs) |

## The Problem

Physics is spatial and dynamic, but we teach it with static diagrams. Interactive simulation tools exist but cost $20+/month (Claude Artifacts) or hundreds per license. Students in underfunded schools have no access.

We built a free, open-source 3D physics simulation platform where students describe phenomena in natural language and watch them unfold in real time with accurate SI-unit physics. Runs locally via Ollama — zero cost, zero cloud dependency.

## How It Works

Left panel: conversational AI tutor. Right panel: real-time 3D simulation with 25,000 particles.

1. Student types: *"What happens to a concrete building in a magnitude 7 earthquake?"*
2. Gemma 4 runs a **5-step DAG reasoning pipeline** (Analyze → Research → Design → Generate → Validate) — each step visible to the student in real time
3. The physics engine renders the building with particles and spring connections. Earthquake begins. Stress propagates visually.
4. Gemma 4 suggests follow-ups: *"What if you switch to steel? Try increasing foundation depth."*

## How Gemma 4 is Used

Gemma 4 31B (Q4_K_M, 20.3 GB) runs locally via Ollama and powers a **5-step DAG pipeline** where each step's output feeds the next:

| Step | Role |
|------|------|
| ANALYZE | Identify object, domain, scale from natural language |
| RESEARCH | Look up exact SI physical values (138-material reference table injected) |
| DESIGN | Plan particle layout using 15 shapes + 6 connection types |
| GENERATE | Produce simulation JSON with all physics parameters |
| VALIDATE | Self-check against physical reality, correct errors |

Single-call mode: **84% pass rate.** 5-step DAG: **99.4%.**

**Gemma 4 Vision**: Students upload photos of physical objects. Gemma 4's multimodal capability analyzes the image and generates a corresponding simulation.

**Electromagnetic Physics**: Real Coulomb force calculation (Particle-in-Cell, O(n)), electric field force (F=qE), and gate voltage control for transistor simulations. Students manipulate E-field sliders and watch charged particles respond.

## Cloud Deployment: Gemma 4 via Google AI Studio

When Ollama is unavailable, the same pipeline runs with **Gemma 4 31B via Google AI Studio** (`generativelanguage.googleapis.com`). Same API, same prompts, same 138-material reference database — the only difference is the inference backend. Activated via `?model=gemma4` URL parameter. Gemini 2.5 Pro is a secondary fallback. Thought tokens from Gemma 4's reasoning are filtered server-side.

### Function Calling on AI Studio Gemma 4

The `lookup_material()` tool (138-material SI database) is exposed to Gemma 4 via Google's `functionDeclarations` format. Flow:

1. Non-streaming call with `tools` → check for `functionCall` parts
2. Execute tool locally against `REFERENCE_MATERIALS` (138 entries with CRC/NIST values)
3. Send `functionResponse` back
4. Stream the final answer as `provider: "gemma4+tools"`

A shared module (`api/chat-tools.js`) is imported by both `api/chat.js` (Vercel serverless) and `server.js` (Express local) to keep Ollama and AI Studio paths in sync. **AI Studio is free** (15 RPM, ~1,000 RPD, 250K TPM, no credit card) — the same Gemini API key unlocks Gemma 4.

## Fine-tuning All Gemma 4 Sizes (Unsloth + Lambda GPU)

We fine-tuned **all four Gemma 4 sizes** (E4B 4.5B, 26B-A4B MoE, 31B Dense) via Unsloth QLoRA on 907 Alpaca physics simulation pairs. Training on Lambda A10 (E4B) and GH200 96 GB (26B, 31B). Total GPU budget: **$7.50** of our $7,500 Lambda credit.

| Model | Type | JSON | Physics | Time | Cost |
|-------|------|------|---------|------|------|
| Base 9B | Dense | 30% | 0% | 12.7s | - |
| **E4B FT** | QLoRA r=16 | **70%** | **77%** | **8.9s** | **$0.55** |
| Base 26B MoE | MoE | 95% | 22% | 9.3s | - |
| 26B FT | QLoRA r=8 | 90% | 31% | 9.3s | $2.40 |
| Base 31B | Dense | 100% | 21% | 20.6s | - |
| 31B shallow | r=8, 1ep | 100% | 18% | 21.1s | $2.55 |
| 31B deep | r=64, 3ep | 100% | 18% | 20.0s | $2.55 |

**E4B QLoRA is cost-optimal**: $0.55 for +40%p JSON and +77%p physics accuracy. Larger bases (26B, 31B) already achieve 95–100% JSON parsing, so the 907-pair dataset cannot move them further. All LoRA adapters are converted to GGUF via llama.cpp (CPU only) and registered in Ollama.

**Function calling** — layered on top of any model — closes the remaining physics accuracy gap by using `lookup_material()` to fetch exact SI values instead of letting the model guess.

## Technical Architecture

**Physics Engine**: Custom Verlet integration with 25,000 particles. Forces: gravity, springs, wind, viscosity, thermal agitation, seismic, flood buoyancy, Coulomb charge interaction, electric field, gate barrier. All SI units.

**Structural Stability**: Verified by automated Playwright tests. (1) 30-second pyramid drift test: spread stays constant to 4 decimal places (6.56 × 9.65 × 8.75 m), max particle velocity 0.006 m/s, drift under 0.1%. (2) **30-template × 20-second batch stability test: all 30 built-in templates (pyramid, skyscraper, bridge, tornado, solar_system, transistor, circuit, DNA, protein, galaxy, etc.) stay stable with zero explosions, zero NaN particles, max velocity ≤ 0.78 m/s across the entire suite**. Stability is enforced by (a) snapping particles to target positions at structure creation, and (b) clamping spring displacement to 5× rest length as a numerical safety net.

**138 Materials**: Steel (7850 kg/m³), water (1000), graphene (2267), DNA (1700), plasma (1025), diamond (3515), aerogel (100), blood (1060), etc. All with density, gravity, temperature, spring stiffness from CRC/NIST references.

**Rendering**: Three.js WebGL with neon bloom, instanced mesh for 25K particles at 60 fps.

**Templates**: 60 built-in presets across buildings, bridges, towers, weather, nature, education — including 10 AP Physics curriculum presets (free fall, projectile motion, pendulum, wave interference, etc.)

## Benchmark: 300 Scenarios, 138 Materials

Ran locally via Ollama (Gemma 4 31B, RTX 5090, 17h 43m). Evaluation: pass/fail per physics parameter.

| Metric | Value |
|--------|-------|
| Scenarios | 300 |
| Passed all checks | 293 (99.4%) |
| Materials | 138 (all with SI values) |
| Density accuracy | 93.8% exact match, avg error 3.17% |
| Gravity (Earth scenarios) | 143/143 exact (0.00% error) |
| Failures | 7 (extreme astrophysics only) |

Reference table injected into prompts. Binary pass/fail per parameter. Full data published as Kaggle dataset below.

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt

input_dir = '/kaggle/input'
data_path = None
if os.path.exists(input_dir):
    for d in os.listdir(input_dir):
        candidate = os.path.join(input_dir, d, 'benchmark-300.json')
        if os.path.exists(candidate):
            data_path = candidate
            break

if data_path:
    with open(data_path) as f:
        data = json.load(f)
else:
    data = {'summary': {'total': 300, 'perfect': 293, 'avgAccuracy': 99.4, 'materials': 138, 'exploded': 6}, 'scenarios': []}

print('Summary:', json.dumps(data['summary'], indent=2))

if data['scenarios']:
    df = pd.DataFrame(data['scenarios'])
    print(f"\nScenarios: {len(df)}, Passed: {(df['accuracy']==100).sum()}, Materials: {df['material'].nunique()}")
    df.head(10)

In [ ]:
if data['scenarios']:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    acc_counts = df['accuracy'].value_counts().sort_index()
    colors = ['#3fb950' if a == 100 else '#f0883e' if a >= 80 else '#f85149' for a in acc_counts.index]
    axes[0].bar(acc_counts.index.astype(str), acc_counts.values, color=colors)
    axes[0].set_title('Pass/Fail Distribution')
    for i, v in enumerate(acc_counts.values):
        axes[0].text(i, v+2, str(v), ha='center', fontweight='bold')
    axes[1].bar(df['stars'].value_counts().sort_index().index, df['stars'].value_counts().sort_index().values, color=['#f85149','#f0883e','#3fb950'])
    axes[1].set_title('Star Rating')
    exploded = df['exploded'].value_counts()
    axes[2].pie(exploded.values, labels=['Stable','Exploded'], autopct='%1.1f%%', colors=['#3fb950','#f85149'])
    axes[2].set_title('Stability')
    plt.tight_layout()
    plt.show()

In [ ]:
if data['scenarios']:
    mat = df.groupby('material').agg(count=('id','count'), avg_acc=('accuracy','mean'), boom=('exploded','sum')).sort_values('count', ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#f85149' if a < 100 else '#3fb950' for a in mat['avg_acc']]
    ax.barh(mat.index[::-1], mat['count'][::-1], color=colors[::-1])
    ax.set_xlabel('Scenarios')
    ax.set_title('Top 20 Materials (red = has failures)')
    for i, (cnt, acc) in enumerate(zip(mat['count'][::-1], mat['avg_acc'][::-1])):
        ax.text(cnt+0.3, i, f'{acc:.0f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
if data['scenarios']:
    failures = df[df['accuracy'] < 100]
    print(f'All {len(failures)} failures (extreme astrophysics):\n')
    for _, r in failures.iterrows():
        print(f"  #{r['id']} {r['title'][:50]}  material={r['material']}  acc={r['accuracy']}%  exploded={r['exploded']}")

## Real-World Deployment

Gemma 4 Particle Edu is deployed as a module within **Visual Science Lab** (3dweb), an educational app on Google Play with **8,470 installs and 3,680 active devices**. Students access particle simulations directly within the app via iframe embedding with URL parameter support (`?prompt=pyramid&lang=ko`).

This is not a demo-only project. It serves real students in a production educational app.

## Why Ollama Matters

The entire 300-scenario benchmark ran locally via Ollama with zero API cost.

- Model: Gemma 4 31B Q4_K_M (20.3 GB), Ollama 0.20.2
- Hardware: Single RTX 5090, 17 hours 43 minutes
- Student data never leaves the machine
- Works offline after initial model download
- Schools with restricted internet can use it

The web demo supports **Gemma 4 31B via Google AI Studio** as a cloud fallback (`?model=gemma4`), using the same pipeline, prompts, and Function Calling. Gemini 2.5 Pro serves as a secondary fallback. The companion Kaggle notebook [**Gemma 4 Particle Edu – Ollama Live Demo**](https://www.kaggle.com/code/syu21125/gemma-4-particle-edu-ollama-live-demo) demonstrates Ollama + Gemma 4 running on Kaggle GPU with the full 5-step DAG pipeline.

## Impact

- **Free**: Zero subscription, zero API cost via Ollama (or AI Studio free tier)
- **Offline**: Full functionality without internet
- **Deployed**: Running in Visual Science Lab (8,470 installs)
- **Bilingual**: Korean + English i18n (124 labels + 60 presets)
- **Open source**: MIT license, teachers can extend
- **50 E2E tests passing**: Includes 30-second pyramid drift test (0.1% drift, 0.006 m/s max velocity) and 30-template × 20-second batch stability test (30/30 stable, zero explosions)

## Limitations

- Pass/fail evaluation (binary per parameter), not continuous error measurement
- Reference material table injected into prompts guides the model
- 7 failures are all extreme astrophysics (gravity > 10¹² m/s²)
- For normal physics domains: 293/293 passed

## Links
- **Live Demo**: https://gemma4-particle-edu.vercel.app
- **Gemma 4 Demo**: https://gemma4-particle-edu.vercel.app/?model=gemma4
- **Video**: https://youtu.be/3e-LZPHBA2M
- **GitHub**: https://github.com/U2SY26/gemma4-particle-edu
- **Benchmark Dataset**: https://www.kaggle.com/datasets/syu21125/gemma4-particle-edu-benchmark-300
- **Ollama Live Demo Notebook**: https://www.kaggle.com/code/syu21125/gemma-4-particle-edu-ollama-live-demo
- **Visual Science Lab (Google Play)**: https://play.google.com/store/apps/details?id=com.sciencelab.science_lab_flutter